# 08 — Train V6.5 trên máy local (teacher YOLO26m → distill YOLO26n cho Raspberry Pi)

**Cách dùng (không cần sửa code):**
1. `git pull` repo, tạo file `.env` ở gốc repo có `CAMBER_API_KEY=...` (xem `.env.example`) và cài Camber CLI
   (`curl -sL https://cli.cambercloud.com/install-v2.sh | bash`, Windows: tải binary, đặt `CAMBER_BIN=` trong `.env`).
2. Cài môi trường: `pip install ultralytics==8.4.172 ncnn pnnx pandas` + torch có CUDA.
3. Mở notebook này → **Run All**. Notebook tự: tải dataset `v6p5_v5` (~5.4 GB) + weights khởi tạo từ Camber,
   chọn batch theo VRAM / workers theo hệ điều hành / cache theo dung lượng ổ, train teacher → student (distill) →
   export NCNN → eval (val gốc + val_clean, MEAN_5) → upload kết quả lên Camber.

- **Thử nhanh trước** (~5–10 phút, 160 ảnh, 2 epoch): đặt `DRY_RUN = True` ở cell 1 (hoặc biến môi trường `DRY_RUN=1`).
- **Bị ngắt / tắt máy / treo**: chạy lại đúng cell đó (hoặc Run All) → tự resume từ checkpoint; xong rồi thì bỏ qua.
  Watchdog: 15 phút không có batch nào chạy xong → tự ngắt và resume (tối đa 3 lần).
- **Dataset** `v6p5_v5` (7 class: insulator, insulator_broken, insulator_flashover, bird_nest, foreign_object,
  broken_strand, person) = bộ chung cho YOLO / RT-DETR / StructDETR; chi tiết `data/label_audit/README.md`.
  Thước đo chính **MEAN_5** = trung bình 5 class lỗi trên `val_clean`; `val` = VAL gốc (so với run cũ).
- Mặc định fine-tune từ model v5 (đã train trên L4: teacher 0.816, student NCNN 0.779 MEAN_5 val_clean).
  `INIT = "coco"` để train lại từ đầu (lâu gấp ~3 lần). Kết quả trên server tốt hơn ở ảnh 800: thử `IMGSZ_TEACHER = 800`.

In [ ]:
# 1. CẤU HÌNH — mặc định chạy được luôn; chỉ sửa nếu muốn thử khác
import os, sys, platform, shutil, time, csv, json, logging
from pathlib import Path

RUN_NAME = "v6p5_v5_local"            # thư mục artifacts/<RUN_NAME>, trên Camber: <RUN_NAME>_<ngày>
DATASET_NAME = "v6p5_v5"
STASH_DATASET = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v5"
STASH_INIT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_v5_20261013"   # teacher/best.pt, student_distill/best.pt
STASH_RESULTS = "stash://bunpmc/projects/yolov11sdi/checkpoints"
INIT = "v5"                           # "v5" = fine-tune từ model v5 (nhanh) | "coco" = từ yolo26m.pt / yolo26n.pt
IMGSZ_TEACHER, IMGSZ_STUDENT = 640, 640
EPOCHS_TEACHER, EPOCHS_STUDENT = (25, 30) if INIT == "v5" else (80, 60)
TRAIN_CLASSES = None                  # None = cả 7 class; ví dụ [0, 1, 2, 3, 4, 5] = bỏ person
UPLOAD_RESULTS = True
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"   # hoặc đặt True để thử nhanh
TARGET_MAP50 = 0.82

# ---- tự động: đường dẫn, phần cứng (không cần sửa) ----
import torch, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
for line in (REPO / ".env").read_text().splitlines() if (REPO / ".env").is_file() else []:   # CAMBER_API_KEY... (không in ra)
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
DATA_DIR = Path(os.environ.get("V6P5_DATA_DIR", REPO / "datasets"))
OUT = REPO / "artifacts" / (RUN_NAME + ("_dryrun" if DRY_RUN else ""))
RUNS, INIT_DIR = OUT / "runs", OUT / "init"
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand", "person"]
DEFECTS = NAMES[1:6]
WINDOWS = platform.system() == "Windows"
CAMBER = os.environ.get("CAMBER_BIN") or shutil.which("camber") or str(Path.home() / ".camber/bin" / ("camber.exe" if WINDOWS else "camber"))

if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); đang có {ultralytics.__version__}: pip install ultralytics==8.4.172")
if torch.cuda.is_available():
    DEVICE, VRAM = 0, torch.cuda.get_device_properties(0).total_memory / 2**30
elif DRY_RUN:
    DEVICE, VRAM = "cpu", 0.0
else:
    raise RuntimeError("Không thấy GPU CUDA (cài torch bản CUDA). Muốn thử trên CPU: DRY_RUN = True")
# batch theo VRAM (teacher YOLO26m; student chạy kèm teacher khi distill nên cũng tốn VRAM)
BATCH_TEACHER = 24 if VRAM >= 20 else 12 if VRAM >= 11 else 6 if VRAM >= 7 else 4
BATCH_STUDENT = 32 if VRAM >= 20 else 16 if VRAM >= 11 else 8
WORKERS = min(4 if WINDOWS else 8, os.cpu_count() or 2)
DATA_DIR.mkdir(parents=True, exist_ok=True)
FREE_GB = shutil.disk_usage(DATA_DIR).free / 2**30
CACHE = "disk" if FREE_GB >= 70 and not DRY_RUN else False   # cache .npy cần ~55 GB; không đủ ổ thì đọc ảnh trực tiếp
if DRY_RUN:
    IMGSZ_TEACHER = IMGSZ_STUDENT = 320; EPOCHS_TEACHER = EPOCHS_STUDENT = 2; BATCH_TEACHER = BATCH_STUDENT = 8; WORKERS = 2
COMMON = dict(device=DEVICE, workers=WORKERS, seed=20261013, deterministic=True, optimizer="AdamW", lr0=1e-3, lrf=0.05,
              cos_lr=True, weight_decay=1e-3, warmup_epochs=1.0 if INIT == "v5" else 3.0, close_mosaic=8 if INIT == "v5" else 15,
              cache=CACHE, amp=True, pretrained=True, plots=False, save_period=5, classes=TRAIN_CLASSES)
print(f"{platform.system()} | GPU: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'} ({VRAM:.0f} GB) | ultralytics {ultralytics.__version__}")
print(f"batch teacher/student {BATCH_TEACHER}/{BATCH_STUDENT} | workers {WORKERS} | cache {CACHE} (ổ trống {FREE_GB:.0f} GB) | "
      f"imgsz {IMGSZ_TEACHER}/{IMGSZ_STUDENT} | epochs {EPOCHS_TEACHER}/{EPOCHS_STUDENT} | INIT {INIT} | DRY_RUN {DRY_RUN}")
print("data:", DATA_DIR, "| kết quả:", OUT, "| camber:", CAMBER if Path(CAMBER).exists() or shutil.which(CAMBER) else "KHÔNG THẤY")

In [ ]:
# 2. TẢI DATASET từ Camber (bỏ qua nếu đã có) + kiểm tra sha256 + sửa đường dẫn trong data.yaml
import hashlib, subprocess, tarfile, random, yaml

def camber_cp(src, dst):
    if not (Path(CAMBER).exists() or shutil.which(CAMBER)):
        raise RuntimeError(f"Không thấy Camber CLI ({CAMBER}). Cài CLI hoặc đặt CAMBER_BIN trong .env; "
                           f"hoặc tự tải {src} về {dst}")
    subprocess.run([CAMBER, "stash", "cp", src, str(dst)], check=True, stdout=subprocess.DEVNULL)

DS = DATA_DIR / DATASET_NAME
if not (DS / "manifest.json").is_file():
    tar = DATA_DIR / f"{DATASET_NAME}.tar"
    for f in (f"{DATASET_NAME}.tar.sha256", f"{DATASET_NAME}.package.json", f"{DATASET_NAME}.tar"):
        if not (DATA_DIR / f).is_file():
            print("tải", f, "..."); camber_cp(f"{STASH_DATASET}/{f}", DATA_DIR / f)
    want = (DATA_DIR / f"{DATASET_NAME}.tar.sha256").read_text().split()[0]
    h = hashlib.sha256()
    with tar.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want:
        tar.unlink(); raise RuntimeError("sha256 không khớp (tải lỗi) → đã xoá file tar, chạy lại cell này")
    print("giải nén ..."); tarfile.open(tar).extractall(DATA_DIR)
    tar.unlink()  # giữ ổ trống cho cache
for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:   # path trong tar là của máy build
    cfg = yaml.safe_load((DS / y).read_text()); cfg.update(path=str(DS), train="images/train", val=v)
    (DS / y).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
assert [yaml.safe_load((DS / "data.yaml").read_text())["names"][i] for i in range(7)] == NAMES
pkg = json.loads((DATA_DIR / f"{DATASET_NAME}.package.json").read_text())
print("dataset OK:", DS, "| files:", pkg["files"], "| labels_sha256:", pkg["labels_sha256"][:16])

if DRY_RUN:   # bộ mini 160/80/80 ảnh để thử toàn bộ notebook nhanh
    M = DATA_DIR / f"{DATASET_NAME}_mini"; random.seed(0)
    if not (M / "data.yaml").is_file():
        for s, n in [("train", 160), ("val", 80), ("val_clean", 80)]:
            (M / "images" / s).mkdir(parents=True, exist_ok=True); (M / "labels" / s).mkdir(parents=True, exist_ok=True)
            for st in random.sample(sorted(p.stem for p in (DS / "labels" / s).glob("*.txt")), n):
                shutil.copy(DS / "images" / s / f"{st}.jpg", M / "images" / s); shutil.copy(DS / "labels" / s / f"{st}.txt", M / "labels" / s)
        for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:
            (M / y).write_text(yaml.safe_dump({"path": str(M), "train": "images/train", "val": v, "names": dict(enumerate(NAMES))}, sort_keys=False))
    DS = M
DATA_YAML, VAL_CLEAN_YAML = DS / "data.yaml", DS / "data_val_clean.yaml"
COMMON["data"] = str(DATA_YAML)
print("train trên:", DATA_YAML)

In [ ]:
# 3. WEIGHTS KHỞI TẠO
INIT_DIR.mkdir(parents=True, exist_ok=True)
if INIT == "v5":
    TEACHER_INIT, STUDENT_INIT = INIT_DIR / "teacher_v5.pt", INIT_DIR / "student_v5.pt"
    for dst, src in [(TEACHER_INIT, "teacher/best.pt"), (STUDENT_INIT, "student_distill/best.pt")]:
        if not dst.is_file():
            print("tải", src); camber_cp(f"{STASH_INIT}/{src}", dst)
else:
    TEACHER_INIT, STUDENT_INIT = "yolo26m.pt", "yolo26n.pt"   # ultralytics tự tải (COCO)
for w in (TEACHER_INIT, STUDENT_INIT):
    print(Path(str(w)).name, "→", YOLO(str(w)).names if INIT == "v5" else "COCO pretrained")

In [ ]:
# 4. HÀM TRAIN: resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, in 1 dòng mỗi epoch
import gc, threading, _thread

def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    final = trainer.stop and getattr(trainer, "_stop_seen", False)   # ultralytics val lại best.pt sau epoch cuối
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  "
          f"lr {trainer.optimizer.param_groups[0]['lr']:.2e}  {time.strftime('%H:%M')}", flush=True)

HANG_MINUTES, MAX_RETRIES = 15, 3

class Watchdog:
    """Ngắt train (như bấm Interrupt) nếu không có batch nào chạy xong trong HANG_MINUTES để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                _thread.interrupt_main()   # chạy được cả Windows (không dùng SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start(); return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    for p in [run / "weights/last.pt", *sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run, best = RUNS / name, RUNS / name / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        targs = (ck or {}).get("train_args", {}) or {}
        if ck is not None and (ck.get("epoch", -1) == -1 or ck["epoch"] + 1 >= targs.get("epochs", 10**9)) \
                and targs.get("epochs", kw.get("epochs")) == kw.get("epochs"):
            print(f"[{name}] đã xong, bỏ qua: {best}"); return best
        if ck is not None and targs.get("epochs") != kw.get("epochs"):
            print(f"[{name}] run cũ {targs.get('epochs')} epoch ≠ {kw.get('epochs')} → bắt đầu lại"); ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir(): shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **{**COMMON, **kw})
        wd = Watchdog(HANG_MINUTES)
        for ev in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(ev, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level; LOGGER.setLevel(logging.WARNING)   # tắt progress bar → output gọn
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}"); return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume."); raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level); del yolo; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [ ]:
# 5. TEACHER — YOLO26m
TEACHER_BEST = train_stage("teacher", TEACHER_INIT, epochs=EPOCHS_TEACHER, batch=BATCH_TEACHER, imgsz=IMGSZ_TEACHER,
                           patience=10 if INIT == "v5" else 20)

In [ ]:
# 6. STUDENT — YOLO26n distill từ teacher (chạy kèm teacher mỗi batch nên chậm hơn ~2 lần so với train thường)
TEACHER_BEST = RUNS / "teacher/weights/best.pt"
assert TEACHER_BEST.is_file(), "Chạy cell 5 (teacher) trước"
STUDENT_BEST = train_stage("student_distill", STUDENT_INIT, epochs=EPOCHS_STUDENT, batch=BATCH_STUDENT, imgsz=IMGSZ_STUDENT,
                           patience=10 if INIT == "v5" else 15, distill_model=str(TEACHER_BEST))

In [ ]:
# 7. EXPORT NCNN cho Raspberry Pi
STUDENT_BEST = RUNS / "student_distill/weights/best.pt"
NCNN_DIR = STUDENT_BEST.with_name("best_ncnn_model")
if not NCNN_DIR.is_dir():
    YOLO(str(STUDENT_BEST)).export(format="ncnn", imgsz=IMGSZ_STUDENT, half=False)
print("Copy thư mục này sang Pi:", NCNN_DIR)
print("Trên Pi: pip install ultralytics ncnn; YOLO('best_ncnn_model', task='detect').predict('anh.jpg', imgsz=%d)" % IMGSZ_STUDENT)

In [ ]:
# 8. EVAL trên val gốc + val_clean → eval.csv (mAP50 + mAP50-95 từng class, MEAN_5 = 5 class lỗi)
import pandas as pd
MODELS = {"teacher": (RUNS / "teacher/weights/best.pt", IMGSZ_TEACHER),
          "student": (RUNS / "student_distill/weights/best.pt", IMGSZ_STUDENT),
          "student_ncnn_pi": (RUNS / "student_distill/weights/best_ncnn_model", IMGSZ_STUDENT)}
rows = []
for vname, vyaml in [("val", DATA_YAML), ("val_clean", VAL_CLEAN_YAML)]:
    for tag, (w, sz) in MODELS.items():
        if not w.exists(): continue
        ncnn_model = w.is_dir()
        b = YOLO(str(w), task="detect").val(data=str(vyaml), imgsz=sz, batch=1 if ncnn_model else 16, plots=False, verbose=False,
                                            device="cpu" if ncnn_model else DEVICE, classes=TRAIN_CLASSES,
                                            project=str(OUT / "eval"), name=f"{tag}_{vname}", exist_ok=True).box
        for i, (c, a50, a) in enumerate(zip(b.ap_class_index, b.ap50, b.ap)):
            rows.append({"model": tag, "val": vname, "class": NAMES[int(c)], "P": float(b.p[i]), "R": float(b.r[i]),
                         "AP50": float(a50), "AP50_95": float(a)})
res = pd.DataFrame(rows)
mean5 = res[res["class"].isin(DEFECTS)].groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean()
res = pd.concat([res, mean5.assign(**{"class": "MEAN_5"})], ignore_index=True)
res.to_csv(OUT / "eval.csv", index=False)
display(res.pivot_table(index=["val", "class"], columns="model", values=["AP50", "AP50_95", "P", "R"], sort=False).round(4))
pi = res.query("model == 'student_ncnn_pi' and val == 'val_clean' and `class` == 'MEAN_5'")
if len(pi):
    v = float(pi.AP50.iloc[0])
    print(f"Model Raspberry Pi (NCNN) val_clean MEAN_5 mAP50 = {v:.4f} →", "ĐẠT" if v > TARGET_MAP50 else "CHƯA ĐẠT", TARGET_MAP50)

In [ ]:
# 9. UPLOAD KẾT QUẢ lên Camber: <STASH_RESULTS>/<RUN_NAME>_<ngày>/
REMOTE = f"{STASH_RESULTS}/{RUN_NAME}_{time.strftime('%Y%m%d')}" + ("_dryrun" if DRY_RUN else "")
files = {f"{s}/{n}": RUNS / s / p for s in ("teacher", "student_distill")
         for n, p in [("best.pt", "weights/best.pt"), ("last.pt", "weights/last.pt"), ("results.csv", "results.csv"), ("args.yaml", "args.yaml")]}
files["eval.csv"] = OUT / "eval.csv"
if UPLOAD_RESULTS:
    for dst, src in files.items():
        if src.is_file():
            camber_cp(str(src), f"{REMOTE}/{dst}"); print("uploaded", dst)
    if NCNN_DIR.is_dir():
        subprocess.run([CAMBER, "stash", "cp", "-r", str(NCNN_DIR), f"{REMOTE}/student_distill/best_ncnn_model"], check=True,
                       stdout=subprocess.DEVNULL); print("uploaded student_distill/best_ncnn_model")
    print("Kết quả trên Camber:", REMOTE)

In [ ]:
# 10. XEM TIẾN ĐỘ (khi mở lại notebook / kernel đã restart; không ảnh hưởng train) — chạy riêng cell 1 rồi cell này
import datetime as dt
for stage in ["teacher", "student_distill"]:
    f = RUNS / stage / "results.csv"
    if not f.exists(): continue
    r_ = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f.open())]
    b_ = max(r_, key=lambda r: float(r["metrics/mAP50(B)"]))
    print(f"== {stage}: {len(r_)} epoch | best mAP50 {float(b_['metrics/mAP50(B)']):.4f} (ep {b_['epoch']}) "
          f"| cập nhật {dt.datetime.fromtimestamp(f.stat().st_mtime):%H:%M:%S}")
    for r in r_[-5:]:
        print(f"  ep {r['epoch']:>3}  mAP50 {float(r['metrics/mAP50(B)']):.4f}  mAP50-95 {float(r['metrics/mAP50-95(B)']):.4f}")